In [2]:
"""
Food Recommendation Pipeline Tester
Input: PhysiologicalDemand (8 chiều) + ConstraintProfile
Output: Ranked dish list + score breakdown
Đọc dữ liệu thật từ SQLite DB
"""

import json
import sqlite3

# ============================================================
# CẤU HÌNH — chỉnh đường dẫn DB ở đây
# ============================================================
DB_PATH = "../cookpad_clean.db"   # <-- sửa thành đường dẫn file .db của bạn


# ============================================================
# LOAD DISHES TỪ SQLITE
# ============================================================
def load_dishes(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cur = conn.cursor()
    cur.execute("""
        SELECT
            id, title,
            dish_thermogenic_score,
            dish_hydration_score,
            dish_warming_score,
            dish_cooling_score,
            dish_satiety_score,
            dish_energy_total,
            dish_sodium_total,
            dish_glycemic_load,
            allergen_summary,
            is_vegan,
            is_vegetarian,
            season_suitability,
            taste_profile,
            cook_time_minutes
        FROM dishes
    """)
    rows = cur.fetchall()
    conn.close()

    dishes = []
    for r in rows:
        try:
            allergens = json.loads(r["allergen_summary"]) if r["allergen_summary"] else []
        except Exception:
            allergens = []

        try:
            season = json.loads(r["season_suitability"]) if r["season_suitability"] else {}
        except Exception:
            season = {}

        try:
            taste = json.loads(r["taste_profile"]) if r["taste_profile"] else {}
        except Exception:
            taste = {}

        dishes.append({
            "id":                     r["id"],
            "title":                  r["title"],
            "dish_thermogenic_score": r["dish_thermogenic_score"] or 0.0,
            "dish_hydration_score":   r["dish_hydration_score"]   or 0.0,
            "dish_warming_score":     r["dish_warming_score"]     or 0.0,
            "dish_cooling_score":     r["dish_cooling_score"]     or 0.0,
            "dish_satiety_score":     r["dish_satiety_score"]     or 0.0,
            "dish_energy_total":      r["dish_energy_total"]      or 0.0,
            "dish_sodium_total":      r["dish_sodium_total"]      or 0.0,
            "dish_glycemic_load":     r["dish_glycemic_load"]     or 0.0,
            "allergen_summary":       allergens,
            "is_vegan":               int(r["is_vegan"] or 0),
            "is_vegetarian":          int(r["is_vegetarian"] or 0),
            "season_suitability":     season,
            "taste_profile":          taste,
            "cook_time_minutes":      r["cook_time_minutes"] or 0,
        })
    return dishes


# ============================================================
# PIPELINE FUNCTIONS
# ============================================================

def apply_constraints(dish, demand, constraints):
    """Hard constraints — trả về lý do loại, hoặc None nếu hợp lệ"""
    if constraints.get("vegan") and not dish["is_vegan"]:
        return "Không phải vegan"
    if constraints.get("vegetarian") and not dish["is_vegetarian"]:
        return "Không phải vegetarian"
    for allergen in constraints.get("allergens", []):
        if allergen in dish["allergen_summary"]:
            return f"Chứa dị ứng: {allergen}"
    if demand["sodium_control_need"] >= 1.0:
        limit = constraints.get("sodium_limit", 600)
        if dish["dish_sodium_total"] > limit:
            return f"Sodium {dish['dish_sodium_total']:.0f}mg > giới hạn {limit}mg"
    if demand["glycemic_control_need"] >= 1.0:
        limit = constraints.get("glycemic_load_limit", 10)
        if dish["dish_glycemic_load"] > limit:
            return f"GL {dish['dish_glycemic_load']:.1f} > giới hạn {limit}"
    return None


def score_dish(dish, demand, season):
    """
    Score(dish) = 0.75 × dot_product(demand, dish_vector)
                + 0.15 × taste_bonus
                + 0.10 × seasonality_factor
    """
    dims = {
        "hydration":   (demand["hydration_need"],    dish["dish_hydration_score"]),
        "electrolyte": (demand["electrolyte_need"],  dish["dish_hydration_score"] * 0.6 + dish["dish_satiety_score"] * 0.4),
        "warming":     (demand["warming_food_need"], dish["dish_warming_score"]),
        "cooling":     (demand["cooling_food_need"], dish["dish_cooling_score"]),
        "thermoreg":   (demand["thermoregulation_need"],
                        dish["dish_warming_score"] if demand["warming_food_need"] >= demand["cooling_food_need"]
                        else dish["dish_cooling_score"]),
    }

    raw_score = sum(d * v for d, v in dims.values()) / len(dims)

    # taste_bonus: dot product với taste_weight của người dùng
    taste_weights = demand.get("taste_weight", {})
    if taste_weights and dish["taste_profile"]:
        taste_bonus = sum(
            taste_weights.get(t, 0.0) * score
            for t, score in dish["taste_profile"].items()
        ) * 0.15
    else:
        taste_avg = sum(dish["taste_profile"].values()) / len(dish["taste_profile"]) if dish["taste_profile"] else 0
        taste_bonus = taste_avg * 0.15

    season_factor = dish["season_suitability"].get(season, 0.5)
    season_bonus  = season_factor * 0.10

    final = 0.75 * raw_score + taste_bonus + season_bonus

    breakdown = {k: round(d * v, 4) for k, (d, v) in dims.items()}
    breakdown["taste_bonus"]  = round(taste_bonus, 4)
    breakdown["season_bonus"] = round(season_bonus, 4)

    return round(final, 4), breakdown


def run_pipeline(demand, constraints, season="summer", dishes=None):
    if dishes is None:
        dishes = []

    passed       = []
    filtered     = []
    seen_titles  = set()   # dedup theo title

    for dish in dishes:
        reason = apply_constraints(dish, demand, constraints)
        if reason:
            filtered.append((dish["title"], reason))
            continue

        # Bỏ qua duplicate title
        if dish["title"] in seen_titles:
            continue
        seen_titles.add(dish["title"])

        score, breakdown = score_dish(dish, demand, season)
        passed.append((dish, score, breakdown))

    passed.sort(key=lambda x: x[1], reverse=True)
    return passed[:10], filtered


# ============================================================
# DISPLAY
# ============================================================

def print_separator(char="─", width=70):
    print(char * width)


def print_results(label, demand, constraints, season, dishes):
    print()
    print_separator("═")
    print(f"  TEST: {label}")
    print_separator("═")

    # Demand summary
    print("\n  📊 PhysiologicalDemand:")
    demand_display = [
        ("thermoregulation_need", demand["thermoregulation_need"]),
        ("hydration_need",        demand["hydration_need"]),
        ("electrolyte_need",      demand["electrolyte_need"]),
        ("warming_food_need",     demand["warming_food_need"]),
        ("cooling_food_need",     demand["cooling_food_need"]),
        ("glycemic_control_need", demand["glycemic_control_need"]),
        ("sodium_control_need",   demand["sodium_control_need"]),
        ("energy_need (kcal)",    demand["energy_need"]),
    ]
    for k, v in demand_display:
        bar = ""
        if isinstance(v, float) and v <= 1.0:
            filled = int(v * 20)
            bar = f"  [{'█' * filled}{'░' * (20 - filled)}]"
        print(f"    {k:<28} {str(v):>8}{bar}")

    # Constraints summary
    print("\n  🔒 ConstraintProfile:")
    print(f"    vegan={constraints.get('vegan', False)}  "
          f"vegetarian={constraints.get('vegetarian', False)}  "
          f"allergens={constraints.get('allergens', [])}  "
          f"season={season}")
    if demand["sodium_control_need"] >= 1.0:
        print(f"    sodium_limit={constraints.get('sodium_limit', 600)}mg  "
              f"gl_limit={constraints.get('glycemic_load_limit', 10)}")

    # Run pipeline
    passed, filtered_out = run_pipeline(demand, constraints, season, dishes=dishes)

    # Filtered summary
    if filtered_out:
        print(f"\n  ❌ Bị loại ({len(filtered_out)} món)")

    # Ranked results
    print(f"\n  ✅ Ranked ({len(passed)} món):\n")
    for i, (dish, score, breakdown) in enumerate(passed, 1):
        medal = ["🥇", "🥈", "🥉"][i - 1] if i <= 3 else f"  {i}."
        print(f"  {medal} [{score:.4f}]  {dish['title']}")
        print(f"       hydration={breakdown['hydration']:.4f}  "
              f"thermoreg={breakdown['thermoreg']:.4f}  "
              f"warming={breakdown['warming']:.4f}  "
              f"cooling={breakdown['cooling']:.4f}")
        print(f"       electrolyte={breakdown['electrolyte']:.4f}  "
              f"taste_bonus={breakdown['taste_bonus']:.4f}  "
              f"season_bonus={breakdown['season_bonus']:.4f}")
        print(f"       Na={dish['dish_sodium_total']:.0f}mg  "
              f"GL={dish['dish_glycemic_load']:.1f}  "
              f"kcal={dish['dish_energy_total']:.0f}  "
              f"⏱{dish['cook_time_minutes']}min  "
              f"allergens={dish['allergen_summary']}")
        print()


# ============================================================
# TEST CASES
# ============================================================

TEST_CASES = [
    {
        "label": "Nắng nóng 38°C · Người khỏe · Mùa hè",
        "season": "summer",
        "demand": {
            "thermoregulation_need": 0.85,
            "hydration_need":        0.90,
            "electrolyte_need":      0.75,
            "warming_food_need":     0.10,
            "cooling_food_need":     0.90,
            "glycemic_control_need": 0.0,
            "sodium_control_need":   0.0,
            "energy_need":           1800,
        },
        "constraints": {"vegan": False, "vegetarian": False, "allergens": []},
    },
    {
        "label": "Lạnh 12°C gió mạnh · Người khỏe · Mùa đông",
        "season": "winter",
        "demand": {
            "thermoregulation_need": 0.88,
            "hydration_need":        0.40,
            "electrolyte_need":      0.30,
            "warming_food_need":     0.90,
            "cooling_food_need":     0.05,
            "glycemic_control_need": 0.0,
            "sodium_control_need":   0.0,
            "energy_need":           2400,
        },
        "constraints": {"vegan": False, "vegetarian": False, "allergens": []},
    },
    {
        "label": "Nắng nóng · Tiểu đường (glycemic control)",
        "season": "summer",
        "demand": {
            "thermoregulation_need": 0.80,
            "hydration_need":        0.85,
            "electrolyte_need":      0.70,
            "warming_food_need":     0.10,
            "cooling_food_need":     0.85,
            "glycemic_control_need": 1.0,
            "sodium_control_need":   0.0,
            "energy_need":           1600,
        },
        "constraints": {
            "vegan": False, "vegetarian": False, "allergens": [],
            "glycemic_load_limit": 100,
        },
    },
    {
        "label": "Bình thường · Tăng huyết áp (sodium control)",
        "season": "spring",
        "demand": {
            "thermoregulation_need": 0.50,
            "hydration_need":        0.55,
            "electrolyte_need":      0.40,
            "warming_food_need":     0.40,
            "cooling_food_need":     0.50,
            "glycemic_control_need": 0.0,
            "sodium_control_need":   1.0,
            "energy_need":           2000,
        },
        "constraints": {
            "vegan": False, "vegetarian": False, "allergens": [],
            "sodium_limit": 600,
        },
    },
    {
        "label": "Nắng nóng · Vegan · Dị ứng đậu nành",
        "season": "summer",
        "demand": {
            "thermoregulation_need": 0.80,
            "hydration_need":        0.88,
            "electrolyte_need":      0.65,
            "warming_food_need":     0.10,
            "cooling_food_need":     0.88,
            "glycemic_control_need": 0.0,
            "sodium_control_need":   0.0,
            "energy_need":           1900,
        },
        "constraints": {
            "vegan": True, "vegetarian": True,
            "allergens": ["soy"],
        },
    },
    {
        "label": "Lạnh · Tiểu đường + Tăng huyết áp (double constraint)",
        "season": "winter",
        "demand": {
            "thermoregulation_need": 0.85,
            "hydration_need":        0.45,
            "electrolyte_need":      0.35,
            "warming_food_need":     0.88,
            "cooling_food_need":     0.08,
            "glycemic_control_need": 1.0,
            "sodium_control_need":   1.0,
            "energy_need":           1800,
        },
        "constraints": {
            "vegan": False, "vegetarian": False, "allergens": [],
            "sodium_limit": 600,
            "glycemic_load_limit": 100,
        },
    },
]


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":
    print("\n" + "=" * 70)
    print("   FOOD RECOMMENDATION PIPELINE TESTER")
    print("   PhysiologicalDemand + ConstraintProfile → Ranked Dishes")
    print("=" * 70)

    try:
        dishes = load_dishes(DB_PATH)
        print(f"\n  ✔ Loaded {len(dishes)} dishes từ {DB_PATH}\n")
    except Exception as e:
        print(f"\n  ✘ Không thể load DB: {e}")
        print(f"    Kiểm tra lại DB_PATH = '{DB_PATH}'\n")
        exit(1)

    for tc in TEST_CASES:
        print_results(
            label=tc["label"],
            demand=tc["demand"],
            constraints=tc["constraints"],
            season=tc["season"],
            dishes=dishes,
        )

    print_separator("═")
    print("  DONE — Review kết quả và điều chỉnh TEST_CASES theo nhu cầu")
    print_separator("═")
    print()


   FOOD RECOMMENDATION PIPELINE TESTER
   PhysiologicalDemand + ConstraintProfile → Ranked Dishes

  ✔ Loaded 8282 dishes từ ../cookpad_clean.db


══════════════════════════════════════════════════════════════════════
  TEST: Nắng nóng 38°C · Người khỏe · Mùa hè
══════════════════════════════════════════════════════════════════════

  📊 PhysiologicalDemand:
    thermoregulation_need            0.85  [█████████████████░░░]
    hydration_need                    0.9  [██████████████████░░]
    electrolyte_need                 0.75  [███████████████░░░░░]
    warming_food_need                 0.1  [██░░░░░░░░░░░░░░░░░░]
    cooling_food_need                 0.9  [██████████████████░░]
    glycemic_control_need             0.0  [░░░░░░░░░░░░░░░░░░░░]
    sodium_control_need               0.0  [░░░░░░░░░░░░░░░░░░░░]
    energy_need (kcal)               1800

  🔒 ConstraintProfile:
    vegan=False  vegetarian=False  allergens=[]  season=summer

  ✅ Ranked (10 món):

  🥇 [0.5842]  Canh khổ qu